# MedGuardAI — Phase 3: DPO (preference / RLHF) on top of the SFT adapter

Direct Preference Optimization continues from the **SFT LoRA adapter** produced by `finetune_gemma_lora.ipynb` and nudges the model toward the *chosen* answers and away from the *rejected* ones in `dpo_pairs.jsonl` (built by `backend/training/build_dpo_dataset.py`).

**Inputs:**
- `dpo_pairs.jsonl` — `{system, prompt, chosen, rejected, weight, meta}` per line.
- the SFT adapter folder `medguard-sft/` (the ~80 MB output of the SFT notebook). If you only kept the merged GGUF, re-run the SFT notebook to regenerate the adapter, or set `SFT_ADAPTER_DIR = None` below to DPO from base Gemma-3-4b instead.

**Outputs:**
- DPO LoRA adapter at `outputs/medguard-dpo/`
- merged GGUF Q4_K_M at `outputs/medguard-dpo-gguf/medguard-gemma-3-4b-dpo-q4_k_m.gguf` — drop-in for LM Studio

**Hardware:** free Colab/Kaggle T4 (16 GB). **Runtime:** ~20–40 min for 1 epoch on a few hundred–few thousand pairs.

> DPO holds *prompt + chosen + rejected* in memory at once (~2× the SFT footprint), so the batch size is 1 with gradient accumulation. If you hit OOM, drop `MAX_LENGTH` to 1024 or restart the runtime to clear fragmentation.

## 1. Install dependencies

Same stack as the SFT notebook — `trl>=0.12` already ships `DPOTrainer`/`DPOConfig`.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.12.0,<0.20.0" peft accelerate bitsandbytes
!pip install datasets sentencepiece protobuf

## 2. Upload the inputs

- Upload `dpo_pairs.jsonl` to `/content/`.
- Upload the SFT adapter folder `medguard-sft/` to `/content/medguard-sft/` (drag the folder into the Files panel, or unzip `medguard-sft.zip` from the SFT run, or download it from your HF Hub repo's `sft-adapter/` path). If you genuinely no longer have it, set `SFT_ADAPTER_DIR = None`.

In [ ]:
import os

DATASET_PATH = "/content/dpo_pairs.jsonl"
SFT_ADAPTER_DIR = "/content/medguard-sft"   # set to None to DPO from base Gemma-3-4b
MAX_SEQ_LENGTH = 1536
MAX_LENGTH = 1536          # full sequence cap for DPO (lower this first if OOM)
MAX_PROMPT_LENGTH = 1024

assert os.path.exists(DATASET_PATH), f"upload dpo_pairs.jsonl to {DATASET_PATH} first"
if SFT_ADAPTER_DIR and not os.path.isdir(SFT_ADAPTER_DIR):
    print(f"WARNING: {SFT_ADAPTER_DIR} not found — falling back to DPO from base Gemma-3-4b")
    SFT_ADAPTER_DIR = None

import json
with open(DATASET_PATH) as f:
    n = sum(1 for _ in f)
print(f"dpo dataset has {n} preference rows")

## 3. Load the model

If `SFT_ADAPTER_DIR` is set, Unsloth loads the base model **with the SFT adapter attached and trainable** (this is the recommended path — DPO continues from where SFT left off). Otherwise it loads stock `gemma-3-4b-it` and attaches a fresh LoRA.

In [ ]:
from unsloth import FastModel
import torch

if SFT_ADAPTER_DIR:
    model, tokenizer = FastModel.from_pretrained(
        model_name=SFT_ADAPTER_DIR,            # Unsloth resolves base + LoRA from the adapter folder
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
    )
    # If your Unsloth build loaded the adapter as non-trainable, uncomment:
    # for n_, p_ in model.named_parameters():
    #     if "lora_" in n_: p_.requires_grad_(True)
else:
    model, tokenizer = FastModel.from_pretrained(
        model_name="unsloth/gemma-3-4b-it-bnb-4bit",
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
    )
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=42,
    )

print("VRAM after loading:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

## 4. Format the preference dataset

`DPOTrainer` wants `prompt` / `chosen` / `rejected` columns. We build the `prompt` with the **Gemma-3 chat template** (same template + `SYSTEM_PROMPT` as the SFT notebook — a mismatch here silently degrades training), and append `<end_of_turn>` to the completions so the model also learns where to stop.

If your TRL version errors on bare-text completions, switch `fmt` to the conversational format (`prompt`/`chosen`/`rejected` as lists of `{"role": ..., "content": ...}` messages) and let DPOTrainer apply the template.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# MUST match SYSTEM_PROMPT in finetune_gemma_lora.ipynb and build_dpo_dataset.py.
SYSTEM_PROMPT = (
    "You are MedGuardAI, a safety-first clinical assistant. Answer based on the "
    "FDA documentation context you are given. If the answer is not supported by "
    "the documentation, say so. Never hallucinate dosages or interactions."
)

raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print("raw fields:", raw.column_names)

EOT = "<end_of_turn>\n"

def fmt(ex):
    sys_text = ex.get("system") or SYSTEM_PROMPT
    convo = [
        {"role": "system", "content": [{"type": "text", "text": sys_text}]},
        {"role": "user", "content": [{"type": "text", "text": ex["prompt"]}]},
    ]
    prompt_text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)
    chosen = ex["chosen"].strip()
    rejected = ex["rejected"].strip()
    if not chosen.endswith("<end_of_turn>"):
        chosen = chosen + " " + EOT.strip()
    if not rejected.endswith("<end_of_turn>"):
        rejected = rejected + " " + EOT.strip()
    return {"prompt": prompt_text, "chosen": chosen, "rejected": rejected}

ds = raw.map(fmt, remove_columns=raw.column_names)
print("\nsample prompt:\n", ds[0]["prompt"][:600])
print("\nsample chosen:\n", ds[0]["chosen"][:300])
print("\nsample rejected:\n", ds[0]["rejected"][:300])

## 5. Train with DPO

T4-tuned: batch size 1 (×2 from the pair) with grad-accum 8, low LR (~10–40× below SFT), `beta=0.1` to anchor to the SFT policy, 1 epoch (DPO overfits fast). `ref_model=None` — with a PEFT model TRL gets the reference policy by disabling the adapter, so there's no second model in VRAM.

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=DPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,
        beta=0.1,
        loss_type="sigmoid",
        max_length=MAX_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        bf16=False, fp16=True,            # T4 has no bf16
        seed=42,
        output_dir="outputs/dpo_ckpt",
        save_strategy="no",
        report_to="none",
    ),
)

stats = trainer.train()
print(stats)

## 6. Save the DPO adapter

In [ ]:
ADAPTER_DIR = "outputs/medguard-dpo"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!du -sh {ADAPTER_DIR}

## 7. Quick inference sanity check

Check that an emergency query still triggers `[EMERGENCY]` and a routine one does not over-escalate.

In [ ]:
for q in [
    "Drug: n/a\nQuestion: I have crushing chest pain radiating to my left arm",
    "Drug: ibuprofen\nQuestion: can I take ibuprofen for a mild headache?",
    "Drug: n/a\nQuestion: my head hurts",
]:
    convo = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": q}]},
    ]
    inputs = tokenizer.apply_chat_template(convo, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=220, do_sample=False)
    print("=" * 80)
    print(q)
    print("-" * 80)
    print(tokenizer.batch_decode(out)[0].split("<start_of_turn>model")[-1])

## 8. Merge to fp16 and convert to GGUF Q4_K_M

Renamed to `medguard-gemma-3-4b-dpo-q4_k_m.gguf` so it sits alongside the SFT GGUF in LM Studio.

In [ ]:
GGUF_DIR = "outputs/medguard-dpo-gguf"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

# Normalize the filename.
import glob, os
for fp in glob.glob(os.path.join(GGUF_DIR, "*.gguf")):
    target = os.path.join(GGUF_DIR, "medguard-gemma-3-4b-dpo-q4_k_m.gguf")
    if os.path.abspath(fp) != os.path.abspath(target):
        os.replace(fp, target)
!ls -la {GGUF_DIR}
!du -sh {GGUF_DIR}

## 9. Download / push to Hugging Face Hub

In [ ]:
# 9a. Direct download
!cd outputs && zip -r medguard-dpo.zip medguard-dpo
!cd outputs && zip -r medguard-dpo-gguf.zip medguard-dpo-gguf
from google.colab import files
files.download("outputs/medguard-dpo.zip")
files.download("outputs/medguard-dpo-gguf.zip")

In [ ]:
# 9b. Push to a private HF Hub repo (optional)
# from huggingface_hub import login, create_repo, upload_folder
# HF_TOKEN = "hf_..."; HF_USER = "your-username"; REPO = f"{HF_USER}/medguardai-gemma-3-4b"
# login(token=HF_TOKEN); create_repo(REPO, private=True, exist_ok=True)
# upload_folder(repo_id=REPO, folder_path=ADAPTER_DIR, path_in_repo="dpo-adapter")
# upload_folder(repo_id=REPO, folder_path=GGUF_DIR, path_in_repo="gguf-dpo")

## 10. Use in MedGuardAI

1. `python backend/training/export_to_lmstudio.py copy --gguf path/to/medguard-gemma-3-4b-dpo-q4_k_m.gguf`
2. In LM Studio, load the new model (keep the base + SFT models loadable too, for the eval comparison).
3. Set `LOCAL_LLM_MODEL` in the repo's `.env` to the DPO model's identifier, restart the backend.
4. `python backend/evaluation/run_eval.py --label dpo` then `python backend/evaluation/compare_runs.py --runs base sft dpo --per-case`.